In [3]:
# -*- coding: utf-8 -*-
"""
Unified 3D gPINN benchmark with strict LHS-based error evaluation and timing.

Protocol aligned with the 3D DAE and 2D PINN benchmarks:
1. Six fixed random seeds.
2. Fixed training points within each seed.
3. A shared 13,000-point LHS test set selected from the reference grid.
4. T_eval uses GPU warmup and repeated forward timing on LHS points only.
5. Error computation, CPU transfer, CSV writing, and loss-history writing are
   excluded from T_train and T_eval.
6. The PDE loss is augmented by first derivatives of the PDE residual with
   respect to x, y, z, and t, with unit gPINN weight.
7. LHS predictions are saved for every seed.
8. For seed 1234 only, the full 51 x 51 x 51 x 51 predicted field is saved
   in chunks for plotting. This extra reconstruction and file writing are
   excluded from T_eval and T_total.
"""

import math
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.spatial import cKDTree
from scipy.stats import qmc
from torch.autograd import grad

# =============================================================================
# 1. Basic settings
# =============================================================================
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
torch.backends.cudnn.benchmark = False

SEEDS = [33, 99, 202, 1234, 5678, 9999]
MU_LIST = [0.01]

X_MIN, X_MAX = -1.0, 1.0
Y_MIN, Y_MAX = -1.0, 1.0
Z_MIN, Z_MAX = -1.0, 1.0
T_FINAL = 0.5

# Training sizes: preserve the original 3D PINN setting.
N_F = 6000
N_B = 6000
N_I = 6000
EPOCHS = 40000
LEARNING_RATE = 1.0e-3
NETWORK_LAYERS = [4, 10, 10, 10, 10, 10, 10, 1]

# Strict LHS test and evaluation timing settings.
NUM_SAMPLES = 13000
LHS_SEED = 1234
EVAL_WARMUP = 20
EVAL_REPEAT = 200

# True means that PINN must reuse the index file already created by DAE.
# This prevents accidental evaluation on a newly generated but different set.
REQUIRE_SHARED_LHS_INDEX = True

BASE_PATH = "."
SAVE_LHS_PREDICTION = True
METHOD_NAME = "gPINN"
USE_GPINN = True
GPINN_WEIGHT = 1.0

# Optional full-field output used only for plotting. It is excluded from all
# benchmark timings and error statistics.
SAVE_FULL_FIELD = True
FULL_FIELD_SEED = 1234
FULL_NT = 51
FULL_NX = 51
FULL_NY = 51
FULL_NZ = 51
FULL_FIELD_CHUNK_SIZE = 200000
EXPECTED_REFERENCE_POINTS = FULL_NT * FULL_NX * FULL_NY * FULL_NZ


# =============================================================================
# 2. Utilities
# =============================================================================
def synchronize_backend():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)


def mean_std(values):
    arr = np.asarray(values, dtype=float)
    if arr.size <= 1:
        return float(np.nanmean(arr)), 0.0
    return float(np.nanmean(arr)), float(np.nanstd(arr, ddof=1))


def compute_error(true_u, pred_u):
    true_vec = np.asarray(true_u, dtype=np.float64).reshape(-1)
    pred_vec = np.asarray(pred_u, dtype=np.float64).reshape(-1)

    if true_vec.shape != pred_vec.shape:
        raise ValueError(
            f"Shape mismatch in error computation: true={true_vec.shape}, "
            f"pred={pred_vec.shape}."
        )

    denom = np.linalg.norm(true_vec)
    if not np.isfinite(denom) or denom <= 0.0:
        raise ValueError("The reference solution has a zero or non-finite L2 norm.")

    diff = pred_vec - true_vec
    e2 = np.linalg.norm(diff) / denom
    einf = np.max(np.abs(diff))
    return float(e2), float(einf)


# =============================================================================
# 3. Network and physical problem
# =============================================================================
class PINN(nn.Module):
    def __init__(self, layers):
        super().__init__()
        self.layers = nn.ModuleList()
        for in_features, out_features in zip(layers[:-1], layers[1:]):
            layer = nn.Linear(in_features, out_features)
            nn.init.xavier_normal_(layer.weight)
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)
            self.layers.append(layer)
        self.activation = nn.Tanh()

    def forward(self, inputs):
        hidden = inputs
        for layer in self.layers[:-1]:
            hidden = self.activation(layer(hidden))
        return self.layers[-1](hidden)


def source_function(x, y, z):
    return (
        torch.cos(math.pi * x)
        * torch.cos(math.pi * y)
        * torch.cos(math.pi * z)
    )


def initial_condition(x, y, z, mu):
    return 3.0 * torch.tanh(x / mu + y + z) - 1.0


def compute_loss(
    model,
    X_f,
    X_b_x_left,
    X_b_x_right,
    X_b_y_bottom,
    X_b_y_top,
    X_b_z_left,
    X_b_z_right,
    X_i,
    U_i_target,
    mu,
    use_gpinn=True,
    gpinn_weight=1.0,
):
    """Return total gPINN loss and detached component values.

    Let
        r = mu * Delta u - u_t + u * (u_x + u_y + u_z) - f.
    The gradient-enhanced term is
        mean(r_x^2 + r_y^2 + r_z^2 + r_t^2).
    With ``gpinn_weight=1`` this exactly preserves the original 3D gPINN
    formulation supplied by the user.
    """
    # PDE residual:
    # mu * Delta u - u_t + u * (u_x + u_y + u_z) - f = 0.
    u_f = model(X_f)
    first_grads = grad(u_f.sum(), X_f, create_graph=True)[0]
    u_x = first_grads[:, 0:1]
    u_y = first_grads[:, 1:2]
    u_z = first_grads[:, 2:3]
    u_t = first_grads[:, 3:4]

    # create_graph=True is required because the residual itself is
    # differentiated once more in the gPINN term.
    u_xx = grad(u_x.sum(), X_f, create_graph=True)[0][:, 0:1]
    u_yy = grad(u_y.sum(), X_f, create_graph=True)[0][:, 1:2]
    u_zz = grad(u_z.sum(), X_f, create_graph=True)[0][:, 2:3]

    f_val = source_function(X_f[:, 0:1], X_f[:, 1:2], X_f[:, 2:3])
    residual = (
        mu * (u_xx + u_yy + u_zz)
        - u_t
        + u_f * (u_x + u_y + u_z)
        - f_val
    )
    loss_pde = torch.mean(residual.square())

    if use_gpinn:
        residual_grads = grad(
            residual.sum(),
            X_f,
            create_graph=True,
            retain_graph=True,
        )[0]
        loss_gpinn = torch.mean(torch.sum(residual_grads.square(), dim=1))
    else:
        loss_gpinn = torch.zeros((), dtype=DTYPE, device=DEVICE)

    # Boundary conditions:
    # u(-1,y,z,t)=-4, u(1,y,z,t)=2,
    # periodicity in y and z is imposed softly on function values.
    u_x_left = model(X_b_x_left)
    u_x_right = model(X_b_x_right)
    u_y_bottom = model(X_b_y_bottom)
    u_y_top = model(X_b_y_top)
    u_z_left = model(X_b_z_left)
    u_z_right = model(X_b_z_right)

    loss_bc = torch.mean(
        (u_x_left + 4.0).square()
        + (u_x_right - 2.0).square()
        + (u_y_bottom - u_y_top).square()
        + (u_z_left - u_z_right).square()
    )

    # Initial condition.
    u_i = model(X_i)
    loss_ic = torch.mean((u_i.reshape(-1) - U_i_target).square())

    total_loss = loss_pde + gpinn_weight * loss_gpinn + loss_bc + loss_ic
    components = {
        "pde": loss_pde.detach(),
        "gpinn": loss_gpinn.detach(),
        "bc": loss_bc.detach(),
        "ic": loss_ic.detach(),
    }
    return total_loss, components


# =============================================================================
# 4. LHS reference-data utilities
# =============================================================================
def get_target_col(df):
    if "u" in df.columns:
        return "u"
    if "u0" in df.columns:
        return "u0"
    return df.columns[-1]


def load_true_solution(mu):
    mu_id = int(round(-math.log10(mu)))
    candidates = [
        f"3d_U0_true_mu{mu:.0e}.csv",
        f"3d_U0_all_t_u_x_y_z_t_mu{mu_id}_51_mathematica_619.csv",
    ]

    # Preserve compatibility with the filename used by the existing 3D DAE code.
    if np.isclose(mu, 1.0e-2):
        candidates.append("3d_U0_all_t_u_x_y_z_t_mu2_51_mathematica_619.csv")

    for filename in candidates:
        path = os.path.join(BASE_PATH, filename)
        if os.path.exists(path):
            df = pd.read_csv(path)
            df.columns = [str(col).lower().strip() for col in df.columns]

            required = {"t", "x", "y", "z"}
            missing = required.difference(df.columns)
            if missing:
                raise ValueError(
                    f"Reference file {filename} is missing columns: {sorted(missing)}"
                )

            df = df.sort_values(by=["t", "x", "y", "z"]).reset_index(drop=True)
            return df, filename

    raise FileNotFoundError(
        "Cannot find the 3D reference solution. Tried: " + ", ".join(candidates)
    )


def build_or_load_lhs_test_set(mu):
    df_true, reference_filename = load_true_solution(mu)
    index_file = f"3d_LHS_sample_indices_mu{mu:.0e}.npy"

    sample_indices = None
    if os.path.exists(index_file):
        loaded = np.asarray(np.load(index_file), dtype=np.int64).reshape(-1)
        valid = (
            loaded.size == NUM_SAMPLES
            and np.unique(loaded).size == NUM_SAMPLES
            and loaded.min() >= 0
            and loaded.max() < len(df_true)
        )
        if valid:
            sample_indices = loaded
            print(f"[mu={mu}] Loaded valid LHS indices from {index_file}.")
        else:
            print(f"[mu={mu}] Existing LHS index file is invalid and will be regenerated.")

    if sample_indices is None and REQUIRE_SHARED_LHS_INDEX:
        raise FileNotFoundError(
            f"The shared DAE/PINN LHS index file {index_file} is missing or invalid. "
            "Run the 3D DAE code once to create it, or set "
            "REQUIRE_SHARED_LHS_INDEX=False to generate it here."
        )

    if sample_indices is None:
        total_points = len(df_true)
        if total_points < NUM_SAMPLES:
            raise ValueError(
                f"Reference grid contains only {total_points} points, "
                f"but NUM_SAMPLES={NUM_SAMPLES}."
            )

        coordinate_columns = ["t", "x", "y", "z"]
        all_points = df_true[coordinate_columns].to_numpy(dtype=np.float64)
        lower = all_points.min(axis=0)
        upper = all_points.max(axis=0)
        kdtree = cKDTree(all_points)

        selected = []
        used = set()
        batch_id = 0

        while len(selected) < NUM_SAMPLES and batch_id < 100:
            sampler = qmc.LatinHypercube(d=4, seed=LHS_SEED + batch_id)
            lhs_unit = sampler.random(n=NUM_SAMPLES)
            lhs_scaled = qmc.scale(lhs_unit, lower, upper)
            _, candidate_indices = kdtree.query(lhs_scaled, k=1)

            for idx in np.asarray(candidate_indices).reshape(-1):
                idx = int(idx)
                if idx not in used:
                    used.add(idx)
                    selected.append(idx)
                    if len(selected) == NUM_SAMPLES:
                        break
            batch_id += 1

        # Deterministic completion in the unlikely event that nearest-neighbour
        # collisions prevent the LHS loop from reaching NUM_SAMPLES.
        if len(selected) < NUM_SAMPLES:
            remaining = np.setdiff1d(
                np.arange(total_points, dtype=np.int64),
                np.asarray(selected, dtype=np.int64),
                assume_unique=False,
            )
            rng = np.random.default_rng(LHS_SEED)
            fill = rng.choice(
                remaining,
                size=NUM_SAMPLES - len(selected),
                replace=False,
            )
            selected.extend(int(idx) for idx in fill)

        sample_indices = np.asarray(selected, dtype=np.int64)
        np.save(index_file, sample_indices)
        print(
            f"[mu={mu}] Generated and saved {NUM_SAMPLES} unique LHS indices "
            f"to {index_file}."
        )

    sampled = df_true.iloc[sample_indices]
    t_np = sampled["t"].to_numpy(dtype=np.float64).reshape(-1, 1)
    x_np = sampled["x"].to_numpy(dtype=np.float64).reshape(-1, 1)
    y_np = sampled["y"].to_numpy(dtype=np.float64).reshape(-1, 1)
    z_np = sampled["z"].to_numpy(dtype=np.float64).reshape(-1, 1)

    true_col = get_target_col(df_true)
    true_np = sampled[true_col].to_numpy(dtype=np.float64).reshape(-1)

    net_input = torch.tensor(
        np.hstack([x_np, y_np, z_np, t_np]),
        dtype=DTYPE,
        device=DEVICE,
    )

    if len(df_true) != EXPECTED_REFERENCE_POINTS:
        raise ValueError(
            f"Reference file {reference_filename} contains {len(df_true)} rows, "
            f"but a 51^4 grid must contain {EXPECTED_REFERENCE_POINTS} rows."
        )

    print(
        f"[mu={mu}] Reference={reference_filename} | "
        f"reference rows={len(df_true)}=51^4 | "
        f"shared index file={index_file} | LHS points={sample_indices.size}"
    )

    return {
        "net_input": net_input,
        "true_lhs_np": true_np,
        "t_np": t_np,
        "x_np": x_np,
        "y_np": y_np,
        "z_np": z_np,
        "n_test": int(sample_indices.size),
        "reference_file": reference_filename,
        "reference_rows": int(len(df_true)),
        "lhs_index_file": index_file,
    }


# =============================================================================
# 5. Full-field reconstruction for plotting
# =============================================================================
def save_full_field_prediction(model, mu, seed):
    """Save the complete 51^4 PINN field in original meshgrid flattening order.

    Row order is identical to
        T, X, Y, Z = np.meshgrid(t_vals, x_vals, y_vals, z_vals, indexing="ij")
        flatten(order="C")
    namely: t is the slowest index and z is the fastest index.

    The prediction is evaluated and written in chunks so that neither the full
    4D input tensor nor the full output vector has to reside in memory.
    This function must be called outside all benchmark timing blocks.
    """
    if seed != FULL_FIELD_SEED:
        return None

    total = FULL_NT * FULL_NX * FULL_NY * FULL_NZ
    t_vals = np.linspace(0.0, T_FINAL, FULL_NT, dtype=np.float64)
    x_vals = np.linspace(X_MIN, X_MAX, FULL_NX, dtype=np.float64)
    y_vals = np.linspace(Y_MIN, Y_MAX, FULL_NY, dtype=np.float64)
    z_vals = np.linspace(Z_MIN, Z_MAX, FULL_NZ, dtype=np.float64)

    filename = f"3d_{METHOD_NAME}_mu{mu:.2f}_U0_predicted_seed{seed}.csv"
    first_chunk = True

    print(
        f"  > Saving full 51^4 prediction for seed={seed}: "
        f"{total} rows -> {filename}"
    )

    model.eval()
    with torch.no_grad():
        for start in range(0, total, FULL_FIELD_CHUNK_SIZE):
            stop = min(start + FULL_FIELD_CHUNK_SIZE, total)
            flat = np.arange(start, stop, dtype=np.int64)

            iz = flat % FULL_NZ
            q = flat // FULL_NZ
            iy = q % FULL_NY
            q //= FULL_NY
            ix = q % FULL_NX
            it = q // FULL_NX

            t_chunk = t_vals[it]
            x_chunk = x_vals[ix]
            y_chunk = y_vals[iy]
            z_chunk = z_vals[iz]

            inputs_np = np.column_stack(
                [x_chunk, y_chunk, z_chunk, t_chunk]
            ).astype(np.float32, copy=False)
            inputs = torch.from_numpy(inputs_np).to(
                device=DEVICE, dtype=DTYPE, non_blocking=False
            )
            u_chunk = model(inputs).reshape(-1).cpu().numpy()

            # Preserve the column names and order used by the original 3D PINN
            # plotting output.
            df_chunk = pd.DataFrame(
                {
                    "u": u_chunk,
                    "x": x_chunk,
                    "y": y_chunk,
                    "z": z_chunk,
                    "t": t_chunk,
                }
            )
            df_chunk.to_csv(
                filename,
                mode="w" if first_chunk else "a",
                header=first_chunk,
                index=False,
            )
            first_chunk = False

    print(f"  > Full-field prediction saved: {filename}")
    return filename


# =============================================================================
# 6. Training-data construction
# =============================================================================
def build_training_data(mu):
    # Interior points.
    x_f = torch.rand(N_F, 1, device=DEVICE, dtype=DTYPE) * (X_MAX - X_MIN) + X_MIN
    y_f = torch.rand(N_F, 1, device=DEVICE, dtype=DTYPE) * (Y_MAX - Y_MIN) + Y_MIN
    z_f = torch.rand(N_F, 1, device=DEVICE, dtype=DTYPE) * (Z_MAX - Z_MIN) + Z_MIN
    t_f = torch.rand(N_F, 1, device=DEVICE, dtype=DTYPE) * T_FINAL
    X_f = torch.cat([x_f, y_f, z_f, t_f], dim=1).requires_grad_(True)

    # A common random parameterization is reused to construct the six faces.
    x_b = torch.rand(N_B, 1, device=DEVICE, dtype=DTYPE) * (X_MAX - X_MIN) + X_MIN
    y_b = torch.rand(N_B, 1, device=DEVICE, dtype=DTYPE) * (Y_MAX - Y_MIN) + Y_MIN
    z_b = torch.rand(N_B, 1, device=DEVICE, dtype=DTYPE) * (Z_MAX - Z_MIN) + Z_MIN
    t_b = torch.rand(N_B, 1, device=DEVICE, dtype=DTYPE) * T_FINAL

    X_b_x_left = torch.cat([torch.full_like(x_b, X_MIN), y_b, z_b, t_b], dim=1)
    X_b_x_right = torch.cat([torch.full_like(x_b, X_MAX), y_b, z_b, t_b], dim=1)
    X_b_y_bottom = torch.cat([x_b, torch.full_like(y_b, Y_MIN), z_b, t_b], dim=1)
    X_b_y_top = torch.cat([x_b, torch.full_like(y_b, Y_MAX), z_b, t_b], dim=1)
    X_b_z_left = torch.cat([x_b, y_b, torch.full_like(z_b, Z_MIN), t_b], dim=1)
    X_b_z_right = torch.cat([x_b, y_b, torch.full_like(z_b, Z_MAX), t_b], dim=1)

    # Initial points.
    x_i = torch.rand(N_I, 1, device=DEVICE, dtype=DTYPE) * (X_MAX - X_MIN) + X_MIN
    y_i = torch.rand(N_I, 1, device=DEVICE, dtype=DTYPE) * (Y_MAX - Y_MIN) + Y_MIN
    z_i = torch.rand(N_I, 1, device=DEVICE, dtype=DTYPE) * (Z_MAX - Z_MIN) + Z_MIN
    t_i = torch.zeros(N_I, 1, device=DEVICE, dtype=DTYPE)
    X_i = torch.cat([x_i, y_i, z_i, t_i], dim=1)
    U_i_target = initial_condition(x_i, y_i, z_i, mu).reshape(-1).detach()

    return {
        "X_f": X_f,
        "X_b_x_left": X_b_x_left,
        "X_b_x_right": X_b_x_right,
        "X_b_y_bottom": X_b_y_bottom,
        "X_b_y_top": X_b_y_top,
        "X_b_z_left": X_b_z_left,
        "X_b_z_right": X_b_z_right,
        "X_i": X_i,
        "U_i_target": U_i_target,
    }


# =============================================================================
# 7. Main benchmark
# =============================================================================
def main():
    print("\n" + "=" * 88)
    print("Starting 3D gPINN benchmark with strict LHS-based T_eval and error")
    print(
        f"Device: {DEVICE} | dtype: {DTYPE} | "
        f"LHS test points: {NUM_SAMPLES} | LHS seed: {LHS_SEED}"
    )
    print(
        f"T_eval warmup: {EVAL_WARMUP} | repeated timing: {EVAL_REPEAT} | "
        f"epochs: {EPOCHS}"
    )
    print(
        f"gPINN enabled: {USE_GPINN} | "
        f"gPINN weight: {GPINN_WEIGHT} | residual-gradient directions: x,y,z,t"
    )
    print("=" * 88 + "\n")

    lhs_data = {mu: build_or_load_lhs_test_set(mu) for mu in MU_LIST}
    metrics = {mu: [] for mu in MU_LIST}

    for mu in MU_LIST:
        print("\n" + "=" * 88)
        print(f"Starting 3D {METHOD_NAME} for mu={mu}")
        print("=" * 88)

        data_mu = lhs_data[mu]
        net_input_eval = data_mu["net_input"]
        true_lhs_np = data_mu["true_lhs_np"]
        n_test = data_mu["n_test"]

        for seed in SEEDS:
            print(f"\n--- Running Seed: {seed} ---")
            set_seed(seed)

            train_data = build_training_data(mu)
            total_points = N_F + 6 * N_B + N_I
            total_point_steps = EPOCHS * total_points

            print(
                "  [Points] "
                f"PDE={N_F} | BC faces=6x{N_B}={6 * N_B} | "
                f"IC={N_I} | total={total_points}"
            )

            model = PINN(NETWORK_LAYERS).to(device=DEVICE, dtype=DTYPE)
            optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
            loss_history_gpu = []

            # -----------------------------------------------------------------
            # Phase A: training time. No CPU transfer or file I/O is included.
            # -----------------------------------------------------------------
            model.train()
            synchronize_backend()
            train_start = time.perf_counter()

            for _ in range(EPOCHS):
                optimizer.zero_grad(set_to_none=True)

                loss, _ = compute_loss(
                    model=model,
                    X_f=train_data["X_f"],
                    X_b_x_left=train_data["X_b_x_left"],
                    X_b_x_right=train_data["X_b_x_right"],
                    X_b_y_bottom=train_data["X_b_y_bottom"],
                    X_b_y_top=train_data["X_b_y_top"],
                    X_b_z_left=train_data["X_b_z_left"],
                    X_b_z_right=train_data["X_b_z_right"],
                    X_i=train_data["X_i"],
                    U_i_target=train_data["U_i_target"],
                    mu=mu,
                    use_gpinn=USE_GPINN,
                    gpinn_weight=GPINN_WEIGHT,
                )
                loss.backward()
                optimizer.step()
                loss_history_gpu.append(loss.detach())

            synchronize_backend()
            T_train = time.perf_counter() - train_start

            # One GPU-to-CPU transfer after training; excluded from T_train.
            loss_history = (
                torch.stack(loss_history_gpu).cpu().numpy().astype(np.float64)
            )
            e_loss = float(loss_history[-1])

            loss_filename = (
                f"3d_{METHOD_NAME}_loss_history_mu{mu:.0e}_seed{seed}.npy"
            )
            np.save(loss_filename, loss_history)

            T_train_per_iter_ms = T_train * 1.0e3 / EPOCHS
            T_train_per_iter_point_us = (
                T_train * 1.0e6 / total_point_steps
            )

            print(
                f"  > Trained: T_train={T_train:.2f}s | "
                f"e_loss={e_loss:.3e} | loss_file={loss_filename}"
            )

            # -----------------------------------------------------------------
            # Phase B: strict LHS forward timing with warmup and repetition.
            # -----------------------------------------------------------------
            model.eval()

            with torch.no_grad():
                for _ in range(EVAL_WARMUP):
                    _ = model(net_input_eval)

            synchronize_backend()
            eval_start = time.perf_counter()

            with torch.no_grad():
                for _ in range(EVAL_REPEAT):
                    _ = model(net_input_eval)

            synchronize_backend()
            T_eval = (time.perf_counter() - eval_start) / EVAL_REPEAT

            # -----------------------------------------------------------------
            # Phase C: error computation and file writing, excluded from timing.
            # -----------------------------------------------------------------
            with torch.no_grad():
                u_pred_lhs = model(net_input_eval).reshape(-1).cpu().numpy()

            e2, einf = compute_error(true_lhs_np, u_pred_lhs)
            T_total = T_train + T_eval

            print(
                f"    -> [mu={mu}] T_eval={T_eval:.6e}s | "
                f"e2={e2:.3e} | einf={einf:.3e} | T_total={T_total:.2f}s"
            )

            if SAVE_LHS_PREDICTION:
                prediction_filename = (
                    f"3d_{METHOD_NAME}_U0_predicted_LHS_"
                    f"mu{mu:.0e}_seed{seed}.csv"
                )
                df_prediction = pd.DataFrame(
                    {
                        "t": data_mu["t_np"].reshape(-1),
                        "x": data_mu["x_np"].reshape(-1),
                        "y": data_mu["y_np"].reshape(-1),
                        "z": data_mu["z_np"].reshape(-1),
                        "u": u_pred_lhs,
                    }
                )
                df_prediction.to_csv(prediction_filename, index=False)

            # Extra plotting output for seed 1234 only. This occurs after
            # T_train, T_eval, T_total, and errors have already been finalized.
            full_field_filename = None
            if SAVE_FULL_FIELD and seed == FULL_FIELD_SEED:
                full_field_filename = save_full_field_prediction(model, mu, seed)

            metrics[mu].append(
                {
                    "Seed": seed,
                    "mu": mu,
                    "N_test": n_test,
                    "e_loss": e_loss,
                    "e2": e2,
                    "einf": einf,
                    "T_train": T_train,
                    "T_eval": T_eval,
                    "T_total": T_total,
                    "T_train_per_iter_ms": T_train_per_iter_ms,
                    "T_train_per_iter_point_us": T_train_per_iter_point_us,
                    "total_trained_steps": EPOCHS,
                    "total_point_steps": total_point_steps,
                    "final_residual_points": total_points,
                    "eval_warmup": EVAL_WARMUP,
                    "eval_repeat": EVAL_REPEAT,
                    "reference_file": data_mu["reference_file"],
                    "reference_rows": data_mu["reference_rows"],
                    "lhs_index_file": data_mu["lhs_index_file"],
                    "full_field_file": full_field_filename,
                }
            )

    # =========================================================================
    # 8. Save per-seed metrics and print mean +/- sample standard deviation.
    # =========================================================================
    print("\n" + "=" * 88)
    print("ALL SEEDS COMPLETED. GENERATING PUBLICATION TABLES...")
    print("=" * 88 + "\n")

    for mu in MU_LIST:
        df_mu = pd.DataFrame(metrics[mu])
        summary_filename = f"3d_{METHOD_NAME}_mu{mu:.0e}_Metrics_Summary.csv"
        df_mu.to_csv(summary_filename, index=False)

        cols = [
            "N_test",
            "e_loss",
            "e2",
            "einf",
            "T_train",
            "T_eval",
            "T_total",
            "T_train_per_iter_ms",
            "T_train_per_iter_point_us",
            "total_trained_steps",
            "total_point_steps",
            "final_residual_points",
        ]
        stats = {col: mean_std(df_mu[col].to_numpy()) for col in cols}

        print(f"### Results for 3D {METHOD_NAME}, mu={mu} [Mean \\pm Sample Std] ###")
        print(f"N_test: {stats['N_test'][0]:.0f} \\pm {stats['N_test'][1]:.0f}")
        print(f"e_loss: {stats['e_loss'][0]:.3e} \\pm {stats['e_loss'][1]:.3e}")
        print(f"e_2: {stats['e2'][0]:.3e} \\pm {stats['e2'][1]:.3e}")
        print(f"e_inf: {stats['einf'][0]:.3e} \\pm {stats['einf'][1]:.3e}")
        print(f"T_train (s): {stats['T_train'][0]:.2f} \\pm {stats['T_train'][1]:.2f}")
        print(f"T_eval (s): {stats['T_eval'][0]:.6e} \\pm {stats['T_eval'][1]:.6e}")
        print(f"T_total (s): {stats['T_total'][0]:.2f} \\pm {stats['T_total'][1]:.2f}")
        print(
            "T_train/iter (ms): "
            f"{stats['T_train_per_iter_ms'][0]:.4f} "
            f"\\pm {stats['T_train_per_iter_ms'][1]:.4f}"
        )
        print(
            "T_train/(iter*pt) (us): "
            f"{stats['T_train_per_iter_point_us'][0]:.4f} "
            f"\\pm {stats['T_train_per_iter_point_us'][1]:.4f}"
        )
        print(
            "Optimization steps: "
            f"{stats['total_trained_steps'][0]:.1f} "
            f"\\pm {stats['total_trained_steps'][1]:.1f}"
        )
        print(
            "Point-iterations: "
            f"{stats['total_point_steps'][0]:.1f} "
            f"\\pm {stats['total_point_steps'][1]:.1f}"
        )
        print(
            "Final residual points: "
            f"{stats['final_residual_points'][0]:.1f} "
            f"\\pm {stats['final_residual_points'][1]:.1f}"
        )
        print(f"Metrics file: {summary_filename}\n")


if __name__ == "__main__":
    main()



Starting 3D gPINN benchmark with strict LHS-based T_eval and error
Device: cuda:0 | dtype: torch.float32 | LHS test points: 13000 | LHS seed: 1234
T_eval warmup: 20 | repeated timing: 200 | epochs: 40000
gPINN enabled: True | gPINN weight: 1.0 | residual-gradient directions: x,y,z,t

[mu=0.01] Loaded valid LHS indices from 3d_LHS_sample_indices_mu1e-02.npy.
[mu=0.01] Reference=3d_U0_all_t_u_x_y_z_t_mu2_51_mathematica_619.csv | reference rows=6765201=51^4 | shared index file=3d_LHS_sample_indices_mu1e-02.npy | LHS points=13000

Starting 3D gPINN for mu=0.01

--- Running Seed: 33 ---
  [Points] PDE=6000 | BC faces=6x6000=36000 | IC=6000 | total=48000
  > Trained: T_train=2206.22s | e_loss=5.285e+00 | loss_file=3d_gPINN_loss_history_mu1e-02_seed33.npy
    -> [mu=0.01] T_eval=3.466955e-04s | e2=4.522e-01 | einf=4.900e+00 | T_total=2206.22s

--- Running Seed: 99 ---
  [Points] PDE=6000 | BC faces=6x6000=36000 | IC=6000 | total=48000
  > Trained: T_train=2230.69s | e_loss=8.248e+00 | loss_f

In [2]:
pip install scipy -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/8e/6d/41991e503e51fc1134502694c5fa7a1671501a17ffa12716a4a9151af3df/scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
Note: you may need to restart the kernel to use updated packages.
